# Day 2 · Notebook 04 — Student Lab 🧪

Six exercises that turn the lecture claims into **your own measurements**. Each exercise has:

* a short goal and the time budget,
* a **TODO** cell with hints (fill in the `...` / commented lines),
* a hidden **✅ Solution** cell — try first, then reveal (click the "…" / eye icon in Jupyter).

| # | Exercise | Time |
|---|---|---|
| 1 | Swap the backbone: ResNet18 vs MobileNetV2 | 20 min |
| 2 | How many layers should we freeze? | 15 min |
| 3 | Calibration data for INT8: size and representativeness | 15 min |
| 4 | Structured vs unstructured pruning | 15 min |
| 5 | Threads vs latency | 10 min |
| 6 | Batch size vs throughput | 10 min |

Prerequisites: `../models/mobilenetv2_cifar10_fp32.pth`, `../models/model_fp32.onnx`, `../models/model_int8.onnx`
(Notebooks 01–03 or `python scripts/train_and_export.py`).

Work in pairs: one person drives, the other predicts the result **before** each cell runs. Write your
predictions down — being wrong is the most instructive part.

## 0. Setup (run once) — shared helpers

In [ ]:
QUICK_RUN = False            # True -> tiny subsets for a smoke test

import warnings
warnings.filterwarnings("ignore", message=r".*torch\.ao\.quantization is deprecated.*")
import copy, io, json, os, tempfile, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import onnxruntime as ort
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights, resnet18, ResNet18_Weights

SEED, IMG_SIZE = 42, 96
DATA_DIR, MODELS_DIR = Path("..") / "data", Path("..") / "models"
OUT_DIR = Path("outputs"); OUT_DIR.mkdir(exist_ok=True)
LAB_TRAIN = 640 if QUICK_RUN else 3000      # training images for the lab experiments
LAB_TEST = 300 if QUICK_RUN else 2000
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
LABELS = json.loads((MODELS_DIR / "labels.json").read_text())
torch.manual_seed(SEED)

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomCrop(IMG_SIZE, padding=IMG_SIZE // 12, padding_mode="reflect"),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
test_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(),
                              transforms.Normalize(MEAN, STD)])

def fixed_subset(n_total, n, seed):
    g = torch.Generator().manual_seed(seed)
    return torch.randperm(n_total, generator=g)[:n].tolist()

train_aug = datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=train_tf)
train_plain = datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=test_tf)
test_full = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=test_tf)
train_loader = DataLoader(Subset(train_aug, fixed_subset(len(train_aug), LAB_TRAIN, SEED)),
                          batch_size=64, shuffle=True, drop_last=True)
test_ds = Subset(test_full, fixed_subset(len(test_full), LAB_TEST, SEED))
test_loader = DataLoader(test_ds, batch_size=200)
X_test = np.stack([test_ds[i][0].numpy() for i in range(len(test_ds))]).astype(np.float32)
y_test = np.array([test_ds[i][1] for i in range(len(test_ds))])


@torch.no_grad()
def evaluate(model, loader=test_loader):
    model.eval(); correct = total = 0
    for x, y in loader:
        correct += (model(x).argmax(1) == y).sum().item(); total += y.numel()
    return correct / total

def fit(model, epochs=1, lr=1e-3, frozen_modules=()):
    '''Train all parameters with requires_grad=True. Modules in `frozen_modules` are kept in eval()
    mode so their BatchNorm statistics stay fixed.'''
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(train_loader),
                                                pct_start=0.15)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    t0 = time.time()
    for _ in range(epochs):
        model.train()
        for m in frozen_modules:
            m.eval()
        for x, y in train_loader:
            opt.zero_grad(set_to_none=True)
            crit(model(x), y).backward()
            opt.step(); sched.step()
    return time.time() - t0

def n_params(model, trainable_only=False):
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)

def size_mb(model):
    buf = io.BytesIO(); torch.save(model.state_dict(), buf); return len(buf.getvalue()) / 1e6

@torch.inference_mode()
def torch_latency(model, batch=1, threads=4, n=50, warmup=10):
    old = torch.get_num_threads(); torch.set_num_threads(threads)
    model.eval(); x = torch.randn(batch, 3, IMG_SIZE, IMG_SIZE)
    for _ in range(warmup):
        model(x)
    t = []
    for _ in range(n):
        t0 = time.perf_counter(); model(x); t.append((time.perf_counter() - t0) * 1000)
    torch.set_num_threads(old)
    return float(np.median(t))

def ort_session(path, threads=4):
    so = ort.SessionOptions(); so.intra_op_num_threads = threads
    return ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])

def ort_latency(sess, x, n=100, warmup=10):
    for _ in range(warmup):
        sess.run(None, {"input": x})
    t = []
    for _ in range(n):
        t0 = time.perf_counter(); sess.run(None, {"input": x}); t.append((time.perf_counter() - t0) * 1000)
    return float(np.median(t))

def ort_accuracy(path):
    s = ort_session(path)
    preds = np.concatenate([s.run(["logits"], {"input": X_test[i:i + 200]})[0].argmax(1)
                            for i in range(0, len(X_test), 200)])
    return float((preds == y_test).mean())

def load_finetuned():
    m = mobilenet_v2(weights=None); m.classifier[1] = nn.Linear(1280, 10)
    m.load_state_dict(torch.load(MODELS_DIR / "mobilenetv2_cifar10_fp32.pth", map_location="cpu",
                                 weights_only=True))
    return m.eval()

finetuned = load_finetuned()
base_acc = evaluate(finetuned)
print(f"lab train images: {LAB_TRAIN} | test images: {LAB_TEST} | fine-tuned model accuracy: {base_acc:.4f}")

The helpers above are all you need: `fit`, `evaluate`, `n_params`, `size_mb`, `torch_latency`,
`ort_session`, `ort_latency`, `ort_accuracy`, `load_finetuned`. Read their docstrings/signatures first.

---
## Exercise 1 — Swap the backbone: ResNet18 vs MobileNetV2 (20 min)

**Goal:** under the *same* budget (1 epoch on `LAB_TRAIN` images, full fine-tuning, lr 1e-3) compare a fresh
ImageNet-pretrained **ResNet18** with a fresh **MobileNetV2**: trainable parameters, size, batch-1 latency,
training time and accuracy.

**Hints**
* `resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)`; its head is `model.fc` (512 → 1000).
* MobileNetV2's head is `model.classifier[1]` (1280 → 1000).
* Predict first: which one is more accurate? Which one is faster? By how much?

In [ ]:
# TODO 1: build both models with a new 10-class head, fine-tune each with fit(model, epochs=1),
#         then collect n_params, size_mb, torch_latency, train time and evaluate() into a DataFrame.
#
# torch.manual_seed(SEED)
# mnv2 = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
# mnv2.classifier[1] = ...
# r18 = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
# r18.fc = ...
# rows = []
# for name, m in [("MobileNetV2", mnv2), ("ResNet18", r18)]:
#     t = fit(m, epochs=1)
#     rows.append({...})
# pd.DataFrame(rows)

### ✅ Solution (reveal after trying)

In [ ]:
rows = []
for name in ["MobileNetV2", "ResNet18"]:
    torch.manual_seed(SEED)
    if name == "MobileNetV2":
        m = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.last_channel, 10)
    else:
        m = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, 10)
    t = fit(m, epochs=1)
    rows.append({"backbone": name, "params [M]": round(n_params(m) / 1e6, 2),
                 "size [MB]": round(size_mb(m), 1), "latency b=1 [ms]": round(torch_latency(m), 2),
                 "train time [s]": round(t), "accuracy": round(evaluate(m), 4)})
ex1 = pd.DataFrame(rows); display(ex1)

**Discuss:**
* ResNet18 has **5× more parameters** (≈45 MB vs 9 MB on disk) and ≈6× more FLOPs
  (≈1.8 vs 0.3 GFLOPs at 224 px).
* But look at the latency column: FLOPs are **not** latency! MobileNetV2 is built from many small, *depthwise*
  convolutions that do few FLOPs per byte of memory moved (memory-bound) and launch many kernels. In PyTorch
  eager on a desktop CPU at batch 1, ResNet18's few large dense convolutions can therefore be about as fast —
  or faster. On phones, with optimised runtimes (ONNX Runtime, TFLite) and under memory/power limits,
  MobileNet's advantage shows much more clearly.
* Size matters independently of speed: download size, RAM, container image, edge storage.

*There is no universally best backbone — only the best trade-off for your hardware and latency/size budget,
and you only find it by measuring on the target.*

---
## Exercise 2 — How many layers should we freeze? (15 min)

**Goal:** starting from ImageNet weights, freeze the first `k` blocks of `model.features` (MobileNetV2 has 19
blocks: 0–18) and train the rest for 1 epoch. Try `k ∈ {19, 14, 7, 0}` (19 = pure feature extraction,
0 = full fine-tuning). Plot accuracy and trainable parameters vs `k`.

**Hints**
* `for p in model.features[:k].parameters(): p.requires_grad = False`
* pass `frozen_modules=[model.features[:k]]` to `fit` so frozen BatchNorm layers keep their statistics.
* Predict: is the curve monotonic? Where is the "knee"?

In [ ]:
# TODO 2:
# results = []
# for k in [19, 14, 7, 0]:
#     torch.manual_seed(SEED)
#     m = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1); m.classifier[1] = nn.Linear(1280, 10)
#     for p in m.features[:k].parameters():
#         p.requires_grad = ...
#     t = fit(m, epochs=1, frozen_modules=[...])
#     results.append({"frozen blocks": k, "trainable": n_params(m, True), "acc": evaluate(m), "time": t})

### ✅ Solution (reveal after trying)

In [ ]:
res2 = []
for k in [19, 14, 7, 0]:
    torch.manual_seed(SEED)
    m = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1); m.classifier[1] = nn.Linear(1280, 10)
    for p in m.features[:k].parameters():
        p.requires_grad = False
    t = fit(m, epochs=1, lr=3e-3 if k == 19 else 1e-3, frozen_modules=[m.features[:k]])
    res2.append({"frozen blocks": k, "trainable params": n_params(m, True), "train time [s]": round(t),
                 "accuracy": round(evaluate(m), 4)})
ex2 = pd.DataFrame(res2); display(ex2)

fig, ax1 = plt.subplots(figsize=(7, 3.5))
ax1.plot(ex2["frozen blocks"], ex2["accuracy"], "o-", color="tab:blue"); ax1.set_ylabel("accuracy", color="tab:blue")
ax1.set_xlabel("number of frozen blocks in model.features"); ax1.invert_xaxis()
ax2 = ax1.twinx(); ax2.bar(ex2["frozen blocks"], ex2["trainable params"] / 1e6, alpha=.25, color="tab:orange", width=1.5)
ax2.set_ylabel("trainable params [M]", color="tab:orange"); plt.title("Freezing depth vs accuracy"); plt.show()

**Discuss:** unfreezing the later (task-specific) blocks gives most of the benefit; the earliest blocks
(edges/colours) are generic, so freezing them costs little accuracy and saves compute and memory. With very
little data, freezing more also reduces over-fitting. Training time drops with more frozen blocks because no
gradients for those weights are needed (the forward pass still runs through them).
With only 3 000 images and one epoch the curve is often **not monotonic** — full fine-tuning can end slightly
below partial freezing because 2.2 M parameters are adapted with very little data (and the numbers vary by a
few tenths of a point between runs). With more data/epochs full fine-tuning usually wins (Notebook 01: 90 %).

---
## Exercise 3 — Calibration data for INT8 (15 min)

**Goal:** quantize `model_fp32.onnx` statically with ONNX Runtime using calibration sets of different
**sizes** (1, 10, 50, 320 correctly preprocessed training images) and several **unrepresentative** sets of
320 images:

* random noise,
* "forgot `Normalize`" — pixels in [0, 1] instead of ImageNet-normalised (a classic pipeline bug),
* "forgot `/255`" — raw pixel values 0–255,
* "night camera" — correctly normalised but 5× darker images.

Measure the INT8 accuracy for each (always on correctly preprocessed test images).

**Hints**
* Reuse the reader pattern from Notebook 03: `get_next()` returns `{"input": batch}` or `None`.
* Run `quant_pre_process` once, then `quantize_static(..., quant_format=QuantFormat.QDQ, per_channel=True,
  activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8)` for each set.
* `train_raw` below gives un-normalised [0, 1] tensors; normalise manually with `(x - mean) / std`.
* Save the quantized files to `OUT_DIR`, not to `../models`!
* Bonus: compare `CalibrationMethod.MinMax` vs `CalibrationMethod.Percentile` on the good 320-image set.

In [ ]:
from onnxruntime.quantization import (CalibrationDataReader, CalibrationMethod, QuantFormat,
                                      QuantType, quantize_static)
from onnxruntime.quantization.shape_inference import quant_pre_process

train_raw = datasets.CIFAR10(DATA_DIR, train=True, download=True,     # NO Normalize -> [0, 1]
                             transform=transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)),
                                                           transforms.ToTensor()]))
mean_np = np.array(MEAN, np.float32)[:, None, None]; std_np = np.array(STD, np.float32)[:, None, None]

# TODO 3:
# class Reader(CalibrationDataReader):
#     def __init__(self, data, bs=32): ...
#     def get_next(self): ...
# prep = OUT_DIR / "lab_prep.onnx"
# quant_pre_process(str(MODELS_DIR / "model_fp32.onnx"), str(prep))
# idx = fixed_subset(len(train_raw), 320, SEED + 1)
# X_raw = np.stack([train_raw[i][0].numpy() for i in idx])      # [0, 1]
# X_cal = (X_raw - mean_np) / std_np                             # correct preprocessing
# calib_sets = {"1 image": X_cal[:1], ..., "noise": ..., "forgot Normalize": X_raw, "forgot /255": X_raw * 255,
#               "night camera": (X_raw * 0.2 - mean_np) / std_np}
# for name, data in calib_sets.items():
#     out = OUT_DIR / f"lab_int8_{name.replace(' ', '_')}.onnx"
#     quantize_static(...)
#     print(name, ort_accuracy(out))

### ✅ Solution (reveal after trying)

In [ ]:
class Reader(CalibrationDataReader):
    def __init__(self, data, bs=32):
        data = data.astype(np.float32)
        self.it = iter([{"input": data[i:i + bs]} for i in range(0, len(data), bs)])
    def get_next(self):
        return next(self.it, None)

prep = OUT_DIR / "lab_prep.onnx"
quant_pre_process(str(MODELS_DIR / "model_fp32.onnx"), str(prep))
idx = fixed_subset(len(train_raw), 320, SEED + 1)
X_raw = np.stack([train_raw[i][0].numpy() for i in idx])            # [0, 1], not normalised
X_cal = (X_raw - mean_np) / std_np                                   # == the inference preprocessing
rng = np.random.default_rng(0)
calib_sets = {
    "1 image": X_cal[:1], "10 images": X_cal[:10], "50 images": X_cal[:50], "320 images": X_cal,
    "320 noise": rng.standard_normal(X_cal.shape),
    "320 forgot Normalize": X_raw,
    "320 forgot /255": X_raw * 255,
    "320 night camera (x0.2)": (X_raw * 0.2 - mean_np) / std_np,
}
rows = [{"calibration set": "FP32 reference (not quantized)", "method": "-",
         "accuracy": round(ort_accuracy(MODELS_DIR / "model_fp32.onnx"), 4)}]
runs = [(n, CalibrationMethod.MinMax) for n in calib_sets] + [("320 images", CalibrationMethod.Percentile)]
for name, method in runs:
    out = OUT_DIR / f"lab_int8_{name.replace(' ', '_').replace('/', '')}_{method.name}.onnx"
    quantize_static(str(prep), str(out), Reader(calib_sets[name]), quant_format=QuantFormat.QDQ,
                    per_channel=True, activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8,
                    calibrate_method=method)
    rows.append({"calibration set": name, "method": method.name, "accuracy": round(ort_accuracy(out), 4)})
ex3 = pd.DataFrame(rows); display(ex3)

**Discuss:**
* Only **activation** ranges come from calibration (weight ranges are read from the weights). With correct
  preprocessing even very few real images give usable ranges for this model — differences between 1 and 320
  images are within noise here — but a tiny set is fragile: one unusual image can set a bad range. Use a few
  hundred images covering all classes.
* Random noise is surprisingly "OK" here — after normalisation it has a similar value range to real images.
* A **preprocessing mismatch** is catastrophic: calibrating on [0, 1] pixels (forgot `Normalize`) or on 0–255
  pixels (forgot `/255`) sets the activation ranges far too narrow or far too wide → heavy clipping or very coarse
  steps on real data → the accuracy collapses although the FP32 model is perfectly fine.
* Distribution shift (dark "night" images) also costs a lot.

**The calibration set must look like production traffic** — same preprocessing code, same kind of images, all
classes. If production data drifts (new camera, night images), re-calibrate (and re-validate).

---
## Exercise 4 — Structured vs unstructured pruning (15 min)

**Goal:** prune 20 % of the weights of every *regular* (`groups == 1`) conv layer of the fine-tuned model
(1) **unstructured** (`prune.l1_unstructured`) and (2) **structured**: whole output channels by L2 norm
(`prune.ln_structured(..., n=2, dim=0)`). Compare sparsity, accuracy and latency.

**Hints**
* `convs = [m for m in model.modules() if isinstance(m, nn.Conv2d) and m.groups == 1]`
* After pruning call `prune.remove(m, "weight")`.
* Structured sparsity: count output channels whose weights are all zero: `(w.flatten(1).abs().sum(1) == 0)`.
* Predict: same sparsity — which one hurts accuracy more? Does either one get faster?

In [ ]:
# TODO 4:
# def prune_convs(model, structured, amount=0.2):
#     m = copy.deepcopy(model)
#     for conv in [...]:
#         if structured: prune.ln_structured(conv, "weight", amount=amount, n=2, dim=0)
#         else:          prune.l1_unstructured(conv, "weight", amount=amount)
#         prune.remove(conv, "weight")
#     return m
# for structured in [False, True]: ...

### ✅ Solution (reveal after trying)

In [ ]:
def prune_convs(model, structured, amount=0.2):
    m = copy.deepcopy(model)
    for conv in [c for c in m.modules() if isinstance(c, nn.Conv2d) and c.groups == 1]:
        if structured:
            prune.ln_structured(conv, "weight", amount=amount, n=2, dim=0)
        else:
            prune.l1_unstructured(conv, "weight", amount=amount)
        prune.remove(conv, "weight")
    return m

def conv_stats(m):
    convs = [c for c in m.modules() if isinstance(c, nn.Conv2d) and c.groups == 1]
    zeros = sum(int((c.weight == 0).sum()) for c in convs); total = sum(c.weight.numel() for c in convs)
    dead = sum(int((c.weight.flatten(1).abs().sum(1) == 0).sum()) for c in convs)
    chans = sum(c.out_channels for c in convs)
    return zeros / total, dead / chans

rows = [{"variant": "fine-tuned (no pruning)", "weight sparsity": 0.0, "dead channels": 0.0,
         "accuracy": round(base_acc, 4), "latency [ms]": round(torch_latency(finetuned), 2)}]
for structured in [False, True]:
    pm = prune_convs(finetuned, structured)
    sp, dead = conv_stats(pm)
    rows.append({"variant": "structured (channels, L2)" if structured else "unstructured (L1)",
                 "weight sparsity": round(sp, 3), "dead channels": round(dead, 3),
                 "accuracy": round(evaluate(pm), 4), "latency [ms]": round(torch_latency(pm), 2)})
ex4 = pd.DataFrame(rows); display(ex4)

**Discuss:**
* Same 20 % sparsity, but structured pruning removes whole filters → much larger accuracy drop without
  fine-tuning (it removes entire features instead of individual unimportant weights).
* **Neither is faster yet**: `torch.nn.utils.prune` only *zeros* weights; the tensors keep their shape. For
  structured pruning you can then physically rebuild smaller layers (and adjust the next layer's input
  channels and BatchNorm) — libraries such as `torch-pruning` automate this dependency tracking. That smaller
  dense network *is* genuinely faster on any hardware — the main reason structured pruning is used in practice.

---
## Exercise 5 — Threads vs latency (10 min)

**Goal:** measure batch-1 latency of (a) PyTorch eager FP32, (b) ONNX Runtime FP32, (c) ONNX Runtime INT8 for
1, 2, 4 and 8 threads.

**Hints:** `torch_latency(model, threads=t)`; `ort_session(path, threads=t)` + `ort_latency(sess, X_test[:1])`.
Predict: does 8 threads give 8× speed-up?

In [ ]:
# TODO 5:
# rows = []
# for t in [1, 2, 4, 8]:
#     rows.append({"threads": t,
#                  "torch fp32": torch_latency(finetuned, threads=t),
#                  "ort fp32": ort_latency(ort_session(MODELS_DIR / "model_fp32.onnx", t), X_test[:1]),
#                  "ort int8": ...})

### ✅ Solution (reveal after trying)

In [ ]:
rows = []
for t in [1, 2, 4, 8]:
    rows.append({"threads": t,
                 "torch fp32 [ms]": round(torch_latency(finetuned, threads=t), 2),
                 "ort fp32 [ms]": round(ort_latency(ort_session(MODELS_DIR / "model_fp32.onnx", t), X_test[:1]), 3),
                 "ort int8 [ms]": round(ort_latency(ort_session(MODELS_DIR / "model_int8.onnx", t), X_test[:1]), 3)})
ex5 = pd.DataFrame(rows).set_index("threads"); display(ex5)
ex5.plot(marker="o", logy=True, figsize=(7, 3.5), title="Batch-1 latency vs threads (log scale)", grid=True)
plt.ylabel("ms"); plt.show()

**Discuss:** a batch-1 forward pass of a tiny model has little parallel work per layer — beyond a few threads the
synchronisation overhead dominates and latency stops improving (or even gets worse, especially on laptops with
mixed performance/efficiency cores or when other processes compete). In a server that handles **many concurrent
requests** it is usually better to give each worker process 1–2 threads and run several workers
(→ Day 3/4: gunicorn workers, container CPU limits).

---
## Exercise 6 — Batch size vs throughput (10 min)

**Goal:** for ONNX Runtime FP32 and INT8 (4 threads) measure latency per *batch* and throughput
(images / second) for batch sizes 1, 8, 32 and 128.

**Hints:** `x = X_test[:bs]`, `lat = ort_latency(sess, x, n=20)`, `throughput = bs / (lat / 1000)`.
Predict: which grows faster, latency or throughput?

In [ ]:
# TODO 6:
# rows = []
# for variant in ["fp32", "int8"]:
#     sess = ort_session(MODELS_DIR / f"model_{variant}.onnx", threads=4)
#     for bs in [1, 8, 32, 128]:
#         lat = ...
#         rows.append({"variant": variant, "batch": bs, "latency_ms": lat, "img_per_s": ...})

### ✅ Solution (reveal after trying)

In [ ]:
rows = []
for variant in ["fp32", "int8"]:
    sess = ort_session(MODELS_DIR / f"model_{variant}.onnx", threads=4)
    for bs in [1, 8, 32, 128]:
        lat = ort_latency(sess, X_test[:bs], n=20, warmup=3)
        rows.append({"variant": variant, "batch": bs, "latency_ms": round(lat, 2),
                     "img_per_s": round(bs / (lat / 1000))})
ex6 = pd.DataFrame(rows); display(ex6.pivot(index="batch", columns="variant", values=["latency_ms", "img_per_s"]))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for v, g in ex6.groupby("variant"):
    axes[0].plot(g["batch"], g["latency_ms"], "o-", label=v)
    axes[1].plot(g["batch"], g["img_per_s"], "o-", label=v)
axes[0].set(xscale="log", yscale="log", xlabel="batch size", ylabel="latency per batch [ms]", title="Latency")
axes[1].set(xscale="log", xlabel="batch size", ylabel="images / s", title="Throughput")
for ax in axes:
    ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

**Discuss:** batching amortises per-call overhead and uses the SIMD units/threads better → throughput rises
steeply, then saturates; latency per request grows. This is the classic **latency–throughput trade-off**:
an interactive API optimises latency (small batches), an offline job optimises throughput (big batches).
Servers such as Triton or TorchServe do *dynamic batching* — briefly queue requests to form a batch.
INT8's advantage often becomes clearer at larger batch sizes.

---
## Wrap-up

Collect your numbers in the results table in the README (size / latency / accuracy) and be ready to answer:

1. Which single optimisation gave the best "accuracy per ms" on your machine?
2. When would you choose QAT over PTQ?
3. Why does 50 % unstructured sparsity not make the model faster, and what would?
4. Which model file will you deploy tomorrow, and why?